In [1]:
# [CELL 1] STUDENT CONFIGURATION

LAST_NAME = 'Elias'
STUDENT_ID = 'TUPM-26-1127'
SEED_NUM = int(STUDENT_ID[-1])
FAVORITE_ARTIST = 'Tubero'

print('Student:', LAST_NAME)
print('Seed Number:', SEED_NUM)
print('Favorite Artist:', FAVORITE_ARTIST)

Student: Elias
Seed Number: 7
Favorite Artist: Tubero


In [2]:
# [CELL 2] DIAGNOSTIC DECORATOR

def record_diagnostic(function):
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        wrapper.last_result = function(*args, **kwargs)
        return wrapper.last_result

    wrapper.calls = 0
    wrapper.last_result = None
    return wrapper

In [3]:
# [CELL 3] READING GENERATION

def identity_number():
    return sum(ord(char) for char in LAST_NAME + FAVORITE_ARTIST) + SEED_NUM

def generate_readings():
    base = identity_number()
    equipment = ('Motor Current', 'Bearing Temperature', 'Vibration')
    readings = []

    for index, name in enumerate(equipment):
        raw_value = base + index * 37
        if index == 0:
            value = 8.0 + raw_value / 32
        elif index == 1:
            value = 35.0 + raw_value / 4
        else:
            value = raw_value / 16
        readings.append({'equipment': name, 'value': round(value, 2)})

    return readings

readings = generate_readings()
print('Generated Readings:', readings)

Generated Readings: [{'equipment': 'Motor Current', 'value': 43.19}, {'equipment': 'Bearing Temperature', 'value': 325.75}, {'equipment': 'Vibration', 'value': 75.0}]


In [4]:
# [CELL 4] VALIDATION AND CALCULATION

def validate_reading(reading):
    if not isinstance(reading, dict):
        raise TypeError('A reading must be a dictionary.')
    if not isinstance(reading.get('equipment'), str):
        raise ValueError('Reading equipment must be a name.')
    if not isinstance(reading.get('value'), (int, float)):
        raise ValueError('Reading value must be numeric.')
    if reading['value'] < 0:
        raise ValueError('Reading value cannot be negative.')

def calculate_reading(reading):
    validate_reading(reading)
    return round(reading['value'] * (1 + SEED_NUM / 100), 2)

def classify_reading(reading, value):
    limits = {'Motor Current': 14, 'Bearing Temperature': 75, 'Vibration': 9}
    limit = limits[reading['equipment']]
    if value > limit:
        return 'CRITICAL'
    if value > limit * 0.85:
        return 'WARNING'
    return 'NORMAL'

In [6]:
# [CELL 5] MODULAR PROCESSING

@record_diagnostic
def process_readings(readings):
    processed = []

    for reading in readings:
        try:
            validate_reading(reading)
            calibrated = calculate_reading(reading)
            status = classify_reading(reading, calibrated)
            processed.append({**reading, 'calibrated': calibrated, 'status': status})
        except (TypeError, ValueError, KeyError) as error:
            processed.append({**reading, 'status': 'INVALID', 'error': str(error)})

    return processed

processed_readings = process_readings(readings)
print('Processed Readings:', processed_readings)

Processed Readings: [{'equipment': 'Motor Current', 'value': 43.19, 'calibrated': 46.21, 'status': 'CRITICAL'}, {'equipment': 'Bearing Temperature', 'value': 325.75, 'calibrated': 348.55, 'status': 'CRITICAL'}, {'equipment': 'Vibration', 'value': 75.0, 'calibrated': 80.25, 'status': 'CRITICAL'}]


In [7]:
# [CELL 6] DIAGNOSTIC SUMMARY

def diagnostic_summary(processed):
    rows = list(processed)
    invalid = sum(row['status'] == 'INVALID' for row in rows)
    abnormal = sum(row['status'] in {'WARNING', 'CRITICAL'} for row in rows)
    overall = 'UNSAFE' if invalid or any(row['status'] == 'CRITICAL' for row in rows) else 'MONITOR' if abnormal else 'NORMAL'

    lines = [
        f'Student: {LAST_NAME} | Seed: {SEED_NUM} | Artist: {FAVORITE_ARTIST}',
        f'Readings processed: {len(rows)}',
        f'Invalid readings: {invalid}',
        f'Abnormal readings: {abnormal}',
        f'Overall equipment status: {overall}',
        f'Diagnostic calls recorded: {process_readings.calls}',
    ]

    for row in rows:
        detail = row.get('error', f"{row.get('calibrated')} units")
        lines.append(f"- {row['equipment']}: {row['status']} ({detail})")

    return '\n'.join(lines)

print('=== EXERCISE 1: EQUIPMENT DIAGNOSTICS ===')
print(diagnostic_summary(processed_readings))

=== EXERCISE 1: EQUIPMENT DIAGNOSTICS ===
Student: Elias | Seed: 7 | Artist: Tubero
Readings processed: 3
Invalid readings: 0
Abnormal readings: 3
Overall equipment status: UNSAFE
Diagnostic calls recorded: 1
- Motor Current: CRITICAL (46.21 units)
- Bearing Temperature: CRITICAL (348.55 units)
- Vibration: CRITICAL (80.25 units)


In [8]:
# [CELL 7] EXCEPTION HANDLING TEST

invalid_demo = process_readings([{'equipment': 'Test Sensor', 'value': 'invalid'}])
print('Invalid-input demonstration:', invalid_demo)

Invalid-input demonstration: [{'equipment': 'Test Sensor', 'value': 'invalid', 'status': 'INVALID', 'error': 'Reading value must be numeric.'}]
